In [ ]:
# ---------------------------------------------------------------
# Notebook 01: TON-IoT Binary Benchmark
# KMeansSMOTE + Classical ML Models
# ---------------------------------------------------------------

import pandas as pd
import numpy as np
from collections import Counter
import warnings
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from imblearn.over_sampling import KMeansSMOTE

warnings.filterwarnings("ignore")

print("\nNotebook 01: TON-IoT Binary Benchmark | KMeansSMOTE + ML Classifiers\n")
start_time = time.time()

# ---------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------

N_SPLITS_CV = 3
RANDOM_STATE = 42

TRAIN_PATH = "/kaggle/input/datasets/gowr1arun/unsw-nb15-and-ton-iot/toniot_network_train_80.csv"
TEST_PATH = "/kaggle/input/datasets/gowr1arun/unsw-nb15-and-ton-iot/toniot_network_test_20.csv"

# ---------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train 80%: {train_df.shape} | Test 20%: {test_df.shape}")

# ---------------------------------------------------------------
# 2. CLEAN + ENCODE
# ---------------------------------------------------------------

def clean_dataframe(df):
    """
    Cleans the TON-IoT dataframe.

    Steps:
    - Drops possible leakage / high-cardinality identity columns.
    - Replaces inf values.
    - Fills missing values.
    - Encodes categorical columns using factorization.

    Note:
    This is acceptable for baseline experimentation.
    Later notebooks use stricter train-test consistent encoding.
    """

    leak_cols = [
        "src_ip",
        "dst_ip",
        "http_uri",
        "ssl_subject",
        "ssl_issuer",
        "timestamp",
        "date",
        "time",
    ]

    df = df.drop(
        columns=[c for c in leak_cols if c in df.columns],
        errors="ignore",
    ).copy()

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    for col in df.columns:
        if df[col].dtype == "object":
            mode_value = df[col].mode()[0]
            df[col].fillna(mode_value, inplace=True)
            df[col] = pd.factorize(df[col])[0]
        else:
            mean_value = df[col].mean()
            df[col].fillna(mean_value, inplace=True)

    return df


train_df = clean_dataframe(train_df)
test_df = clean_dataframe(test_df)

# ---------------------------------------------------------------
# 3. SPLIT FEATURES / TARGET
# ---------------------------------------------------------------

# Binary classification uses the `label` column:
# 0 = normal
# 1 = attack

X_train_full = train_df.drop(columns=["label", "type"], errors="ignore")
y_train_full = train_df["label"].astype(int).values

X_test_full = test_df.drop(columns=["label", "type"], errors="ignore")
y_test_full = test_df["label"].astype(int).values

print("\nBinary target column: label")
print("Class distribution train:", Counter(y_train_full))
print("Class distribution test :", Counter(y_test_full))

# ---------------------------------------------------------------
# 4. DEFINE MODELS
# ---------------------------------------------------------------

MODELS = {
    "XGBoost": XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        solver="lbfgs",
        n_jobs=-1,
    ),

    "K-NN": KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1,
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
    ),
}

# ---------------------------------------------------------------
# 5. CROSS-VALIDATION + FINAL TEST BENCHMARK
# ---------------------------------------------------------------

all_results = []

for model_name, base_clf in MODELS.items():
    print("\n" + "=" * 80)
    print(f"{model_name} → 3-Fold Cross-Validation")
    print("=" * 80)

    cv = StratifiedKFold(
        n_splits=N_SPLITS_CV,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    cv_acc = []
    cv_prec = []
    cv_rec = []
    cv_f1 = []
    cv_weighted_f1 = []
    cv_auc = []

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train_full, y_train_full), 1):
        X_tr = X_train_full.iloc[tr_idx]
        X_val = X_train_full.iloc[val_idx]

        y_tr = y_train_full[tr_idx]
        y_val = y_train_full[val_idx]

        # Scaling is important for KMeansSMOTE.
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)

        # KMeansSMOTE is applied only on the training fold.
        sampler = KMeansSMOTE(random_state=RANDOM_STATE)
        X_tr_bal, y_tr_bal = sampler.fit_resample(X_tr_scaled, y_tr)

        clf = base_clf.__class__(**base_clf.get_params())
        clf.fit(X_tr_bal, y_tr_bal)

        y_pred = clf.predict(X_val_scaled)
        y_prob = clf.predict_proba(X_val_scaled)[:, 1]

        fold_acc = accuracy_score(y_val, y_pred)
        fold_prec = precision_score(y_val, y_pred, zero_division=0)
        fold_rec = recall_score(y_val, y_pred, zero_division=0)
        fold_f1 = f1_score(y_val, y_pred, zero_division=0)
        fold_weighted_f1 = f1_score(y_val, y_pred, average="weighted", zero_division=0)
        fold_auc = roc_auc_score(y_val, y_prob)

        cv_acc.append(fold_acc)
        cv_prec.append(fold_prec)
        cv_rec.append(fold_rec)
        cv_f1.append(fold_f1)
        cv_weighted_f1.append(fold_weighted_f1)
        cv_auc.append(fold_auc)

        print(
            f"Fold {fold}: "
            f"Acc={fold_acc:.4f} | "
            f"Prec={fold_prec:.4f} | "
            f"Rec={fold_rec:.4f} | "
            f"F1={fold_f1:.4f} | "
            f"Weighted_F1={fold_weighted_f1:.4f} | "
            f"AUC={fold_auc:.4f}"
        )

    print(
        f"\n{model_name} CV Avg: "
        f"Acc={np.mean(cv_acc):.4f} | "
        f"Prec={np.mean(cv_prec):.4f} | "
        f"Rec={np.mean(cv_rec):.4f} | "
        f"F1={np.mean(cv_f1):.4f} | "
        f"Weighted_F1={np.mean(cv_weighted_f1):.4f} | "
        f"AUC={np.mean(cv_auc):.4f}"
    )

    # -----------------------------------------------------------
    # FINAL TRAIN ON FULL TRAIN SET + TEST ON HELD-OUT TEST SET
    # -----------------------------------------------------------

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_full)
    X_test_scaled = scaler.transform(X_test_full)

    sampler = KMeansSMOTE(random_state=RANDOM_STATE)
    X_train_bal, y_train_bal = sampler.fit_resample(X_train_scaled, y_train_full)

    final_clf = base_clf.__class__(**base_clf.get_params())
    final_clf.fit(X_train_bal, y_train_bal)

    y_test_pred = final_clf.predict(X_test_scaled)
    y_test_prob = final_clf.predict_proba(X_test_scaled)[:, 1]

    test_acc = accuracy_score(y_test_full, y_test_pred)
    test_prec = precision_score(y_test_full, y_test_pred, zero_division=0)
    test_rec = recall_score(y_test_full, y_test_pred, zero_division=0)
    test_f1 = f1_score(y_test_full, y_test_pred, zero_division=0)
    test_weighted_f1 = f1_score(y_test_full, y_test_pred, average="weighted", zero_division=0)
    test_auc = roc_auc_score(y_test_full, y_test_prob)

    print(
        f"\n{model_name} Test: "
        f"Acc={test_acc:.4f} | "
        f"Prec={test_prec:.4f} | "
        f"Rec={test_rec:.4f} | "
        f"F1={test_f1:.4f} | "
        f"Weighted_F1={test_weighted_f1:.4f} | "
        f"AUC={test_auc:.4f}"
    )

    all_results.append({
        "Model": model_name,
        "CV Accuracy": np.mean(cv_acc),
        "CV Precision": np.mean(cv_prec),
        "CV Recall": np.mean(cv_rec),
        "CV F1": np.mean(cv_f1),
        "CV Weighted F1": np.mean(cv_weighted_f1),
        "CV AUC": np.mean(cv_auc),
        "Test Accuracy": test_acc,
        "Test Precision": test_prec,
        "Test Recall": test_rec,
        "Test F1": test_f1,
        "Test Weighted F1": test_weighted_f1,
        "Test AUC": test_auc,
    })

# ---------------------------------------------------------------
# 6. RESULTS SUMMARY
# ---------------------------------------------------------------

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    by=["Test F1", "Test Accuracy", "Test AUC"],
    ascending=False,
).reset_index(drop=True)

print("\n" + "=" * 80)
print("NOTEBOOK 01 SUMMARY: TON-IoT Binary Benchmark")
print("=" * 80)

display(results_df)

best_model = results_df.iloc[0]

print("\nBest binary model:")
print(f"Model            : {best_model['Model']}")
print(f"Test Accuracy    : {best_model['Test Accuracy']:.4f}")
print(f"Test F1          : {best_model['Test F1']:.4f}")
print(f"Test Weighted F1 : {best_model['Test Weighted F1']:.4f}")
print(f"Test AUC         : {best_model['Test AUC']:.4f}")

# ---------------------------------------------------------------
# 7. SAVE RESULTS
# ---------------------------------------------------------------

output_path = "/kaggle/working/01_toniot_binary_baseline_results.csv"
results_df.to_csv(output_path, index=False)

print(f"\nSaved binary benchmark results to: {output_path}")

print(f"\nBenchmark complete in {time.time() - start_time:.2f} seconds.\n")

# ---------------------------------------------------------------
# 8. NOTEBOOK CONCLUSION
# ---------------------------------------------------------------

print("""
Notebook 01 conclusion:
Binary TON-IoT detection using the `label` column is nearly saturated.
The strongest models achieve almost perfect normal-vs-attack separation.
Therefore, the main research contribution moves to true multi-class rare-class detection using the `type` column, especially MITM.
""")